# KDD 01 — Làm sạch dữ liệu

## 1. Công đoạn và mục tiêu
Kiểm tra kiểu dữ liệu, làm sạch khoảng trắng ở nhãn ngành và đánh giá các vector câu trả lời lặp. Giữ nguyên workbook và DataFrame gốc; không tự điền giá trị thiếu, gộp ngành, xóa mẫu, chuẩn hóa, chia tập hay huấn luyện.

## 2. Bảo vệ dữ liệu
Chỉ hiển thị thống kê tổng hợp. Không hiển thị tên, mã sinh viên, dấu thời gian, dòng trả lời hoặc vector câu trả lời nguyên vẹn.

In [6]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

assert Path(sys.executable).resolve() == Path(r'C:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe').resolve()
print(f'Kernel sys.executable: {sys.executable}')

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'data_AI_1000_rows.xlsx').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy workbook từ thư mục hiện tại hoặc thư mục cha.')

workbook_path = ROOT / 'data_AI_1000_rows.xlsx'
sheet_name = 'AUGMENTED_RAW_1000'
excel_file = pd.ExcelFile(workbook_path)
assert excel_file.sheet_names == [sheet_name]

df_original = pd.read_excel(workbook_path, sheet_name=sheet_name)
metadata_columns = ['Dấu thời gian', 'Tên Sinh viên', 'Ngành học', 'Giới tính']
label_column = 'Ngành học'
question_columns = list(df_original.columns[4:])

assert df_original.shape == (1000, 58)
assert list(df_original.columns[:4]) == metadata_columns
assert len(question_columns) == 54
assert label_column not in question_columns

print(f'Worksheet: {sheet_name}')
print(f'Dữ liệu gốc: {len(df_original)} dòng × {len(df_original.columns)} cột')
print(f'Số câu hỏi: {len(question_columns)}; cột nhãn: {label_column}')

Kernel sys.executable: c:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe
Worksheet: AUGMENTED_RAW_1000
Dữ liệu gốc: 1000 dòng × 58 cột
Số câu hỏi: 54; cột nhãn: Ngành học


## 3. Kiểm tra dữ liệu và kiểu câu trả lời
Kiểm tra dtype, kiểu giá trị, số ô thiếu và số giá trị phân biệt của từng câu. Các nhãn Likert văn bản đã được xác nhận ở Notebook 00; giữ nguyên chúng nếu workbook không lưu câu trả lời dưới dạng số nguyên 1–5.

In [7]:
answer_type_rows = []
for column in question_columns:
    value_types = sorted({type(value).__name__ for value in df_original[column].dropna()})
    answer_type_rows.append({
        'Câu hỏi': column,
        'dtype pandas': str(df_original[column].dtype),
        'Kiểu giá trị': ', '.join(value_types),
        'Ô thiếu': int(df_original[column].isna().sum()),
        'Giá trị phân biệt': int(df_original[column].nunique(dropna=True)),
    })
answer_type_table = pd.DataFrame(answer_type_rows)

expected_answer_labels = [
    'Chưa bao giờ đúng',
    'Chỉ đúng 1 vài trường hợp',
    'Chỉ đúng 1 nửa',
    'Gần như là đúng',
    'Hoàn toàn đúng',
]
answer_cells = df_original[question_columns]
missing_answers = int(answer_cells.isna().sum().sum())
unknown_answers = int((answer_cells.notna() & ~answer_cells.isin(expected_answer_labels)).sum().sum())
numeric_answers = answer_cells.apply(pd.to_numeric, errors='coerce')
all_answers_are_numeric = bool(numeric_answers.notna().all().all())
numeric_values_are_valid = bool(
    all_answers_are_numeric
    and numeric_answers.ge(1).all().all()
    and numeric_answers.le(5).all().all()
    and numeric_answers.mod(1).eq(0).all().all()
)

display(answer_type_table)
print(f'Ô thiếu trong X: {missing_answers}')
print(f'Câu trả lời ngoài 5 nhãn Likert đã xác nhận: {unknown_answers}')
print(f'Tất cả câu trả lời đã là số nguyên 1–5: {numeric_values_are_valid}')

,Câu hỏi,dtype pandas,Kiểu giá trị,Ô thiếu,Giá trị phân biệt
0,1. Tôi có tính tự lập,str,str,0,5
1,2. Tôi suy nghĩ thực tế,str,str,0,5
2,3. Tôi là người thích nghi với môi trường mới,str,str,0,5
3,"4. Tôi có thể vận hành, điều khiển các máy móc...",str,str,0,5
4,5. Tôi làm các công việc thủ công như gấp giấy...,str,str,0,5
5,6. Tôi thích tiếp xúc với thiên nhiên động vật...,str,str,0,5
6,7. Tôi thích những công việc sử dụng tay chân ...,str,str,0,5
7,8. Tôi thích những công việc thấy ngay kết quả,str,str,0,5
8,9. Tôi thích những công việc ngoài trời hơn là...,str,str,0,5
9,10. Tôi có tìm hiểu khám phá nhiều vấn đề mới,str,str,0,5


Ô thiếu trong X: 0
Câu trả lời ngoài 5 nhãn Likert đã xác nhận: 0
Tất cả câu trả lời đã là số nguyên 1–5: False


## 4. Làm sạch và kiểm tra nhãn ngành
Chỉ bỏ khoảng trắng ở đầu/cuối nhãn `Ngành học`. Không đổi chữ hoa/thường, dấu tiếng Việt hay tên gần giống; các biến thể được báo cáo để chủ dự án quyết định.

In [8]:
y_original = df_original[label_column].astype('string')
y_trimmed = y_original.str.strip()
trimmed_label_rows = int((y_original != y_trimmed).fillna(False).sum())

label_forms = pd.DataFrame({'Tên gốc': y_original, 'Tên sau trim': y_trimmed})
label_forms = label_forms.drop_duplicates()
label_forms['Khóa không phân biệt hoa thường'] = label_forms['Tên sau trim'].str.casefold()
label_variants = (
    label_forms.groupby('Khóa không phân biệt hoa thường', dropna=False)
    .agg(
        Số_cách_viết=('Tên sau trim', 'nunique'),
        Các_tên=('Tên sau trim', lambda values: ' | '.join(sorted(values.dropna().unique()))),
    )
    .reset_index(drop=True)
)
label_variants = label_variants[label_variants['Số_cách_viết'] > 1]

label_count_before = y_original.value_counts(dropna=False).rename('Trước làm sạch')
label_count_after = y_trimmed.value_counts(dropna=False).rename('Sau làm sạch')
label_count_comparison = pd.concat([label_count_before, label_count_after], axis=1).fillna(0).astype(int)
label_count_comparison['Chênh lệch'] = label_count_comparison['Sau làm sạch'] - label_count_comparison['Trước làm sạch']
label_count_comparison = label_count_comparison.sort_values('Sau làm sạch', ascending=False, kind='stable')

print(f'Số hàng có khoảng trắng đầu/cuối được loại bỏ: {trimmed_label_rows}')
print(f'Số nhãn phân biệt trước/sau: {y_original.nunique(dropna=True)}/{y_trimmed.nunique(dropna=True)}')
print('Các nhóm tên khác nhau chỉ bởi chữ hoa/thường sau khi trim:')
display(label_variants)
display(label_count_comparison)

Số hàng có khoảng trắng đầu/cuối được loại bỏ: 0
Số nhãn phân biệt trước/sau: 33/33
Các nhóm tên khác nhau chỉ bởi chữ hoa/thường sau khi trim:


,Số_cách_viết,Các_tên


,Trước làm sạch,Sau làm sạch,Chênh lệch
Ngành học,,,
Công nghệ thông tin,73,73,0
Giáo dục tiểu học,46,46,0
Ngôn ngữ anh,44,44,0
Luật,40,40,0
Kế toán,29,29,0
Giáo dục chính trị,28,28,0
Kỹ thuật phần mềm,28,28,0
Sư phạm ngữ văn,28,28,0
Quản lý đất đai,28,28,0


## 5. Kiểm tra vector câu trả lời trùng
So sánh các nhóm trùng trên đúng 54 câu hỏi và thống kê số nhãn ngành khác nhau trong mỗi nhóm. Không hiển thị vector đáp án và không tự xóa mẫu trùng.

In [9]:
duplicate_answer_mask = answer_cells.duplicated(keep=False)
duplicate_rows = df_original.loc[duplicate_answer_mask, question_columns + [label_column]].copy()
duplicate_groups = (
    duplicate_rows.groupby(question_columns, dropna=False)
    .agg(
        Số_bản_ghi=(label_column, 'size'),
        Số_nhãn_ngành=(label_column, 'nunique'),
        Các_nhãn_ngành=(label_column, lambda values: ' | '.join(sorted(values.astype(str).unique()))),
    )
    .reset_index(drop=True)
)
conflicting_duplicate_groups = duplicate_groups[duplicate_groups['Số_nhãn_ngành'] > 1]

print(f'Số hàng thuộc nhóm có vector trả lời lặp: {int(duplicate_answer_mask.sum())}')
print(f'Số nhóm vector lặp: {len(duplicate_groups)}')
print(f'Số nhóm vector lặp có nhãn ngành mâu thuẫn: {len(conflicting_duplicate_groups)}')
display(duplicate_groups)

if not conflicting_duplicate_groups.empty:
    print('DỪNG: X trùng nhưng y khác. Không tạo dữ liệu sạch; cần chủ dự án quyết định cách xử lý.')
else:
    print('Không phát hiện mâu thuẫn X/y trong các nhóm vector trả lời lặp.')

Số hàng thuộc nhóm có vector trả lời lặp: 4
Số nhóm vector lặp: 2
Số nhóm vector lặp có nhãn ngành mâu thuẫn: 0


,Số_bản_ghi,Số_nhãn_ngành,Các_nhãn_ngành
0,2,1,Quản trị khách sạn
1,2,1,Công nghệ thông tin


Không phát hiện mâu thuẫn X/y trong các nhóm vector trả lời lặp.


## 6. Tạo dữ liệu sạch trong bộ nhớ
Không xóa mẫu trùng. Giữ nguyên 54 câu trả lời và chỉ dùng nhãn đã trim. `X` không chứa nhãn, tên, mã, timestamp, giới tính hay metadata khác. Không xuất dữ liệu sạch ra tệp.

In [10]:
if not conflicting_duplicate_groups.empty:
    raise RuntimeError('Có nhóm X trùng nhưng y khác; dừng trước khi tạo dữ liệu sạch.')
if missing_answers or unknown_answers:
    raise RuntimeError('X có giá trị thiếu hoặc câu trả lời ngoài miền đã xác nhận; cần quyết định trước khi làm sạch.')
if y_trimmed.isna().any() or y_trimmed.eq('').any():
    raise RuntimeError('Nhãn ngành bị thiếu hoặc rỗng sau khi trim; không tự điền nhãn.')

if numeric_values_are_valid:
    cleaned_answers = numeric_answers.astype('int64')
    answer_handling = 'Đã chuyển câu trả lời số nguyên 1–5 sang int64.'
else:
    cleaned_answers = answer_cells.copy()
    answer_handling = 'Giữ nguyên 5 nhãn Likert dạng văn bản; không cần ép kiểu số.'

df_clean = cleaned_answers.copy()
df_clean[label_column] = y_trimmed
X = df_clean[question_columns].copy()
y = df_clean[label_column].copy()

assert len(df_clean) == len(df_original)
assert X.shape == (len(df_original), 54)
assert list(X.columns) == question_columns
assert label_column not in X.columns
assert not set(metadata_columns).intersection(X.columns)
assert len(y) == len(X)

before_after_summary = pd.DataFrame([
    {'Chỉ số': 'Số bản ghi', 'Trước': len(df_original), 'Sau': len(df_clean)},
    {'Chỉ số': 'Số đặc trưng X', 'Trước': len(question_columns), 'Sau': X.shape[1]},
    {'Chỉ số': 'Số nhãn ngành y', 'Trước': y_original.nunique(), 'Sau': y.nunique()},
    {'Chỉ số': 'Số hàng có nhãn được trim', 'Trước': 0, 'Sau': trimmed_label_rows},
    {'Chỉ số': 'Số hàng bị xóa', 'Trước': 0, 'Sau': len(df_original) - len(df_clean)},
], columns=['Chỉ số', 'Trước', 'Sau'])

print(answer_handling)
display(before_after_summary)
display(label_count_comparison)

Giữ nguyên 5 nhãn Likert dạng văn bản; không cần ép kiểu số.


,Chỉ số,Trước,Sau
0,Số bản ghi,1000,1000
1,Số đặc trưng X,54,54
2,Số nhãn ngành y,33,33
3,Số hàng có nhãn được trim,0,0
4,Số hàng bị xóa,0,0


,Trước làm sạch,Sau làm sạch,Chênh lệch
Ngành học,,,
Công nghệ thông tin,73,73,0
Giáo dục tiểu học,46,46,0
Ngôn ngữ anh,44,44,0
Luật,40,40,0
Kế toán,29,29,0
Giáo dục chính trị,28,28,0
Kỹ thuật phần mềm,28,28,0
Sư phạm ngữ văn,28,28,0
Quản lý đất đai,28,28,0


## 7. Nhận xét và kết luận
Các bảng trên so sánh phân bố ngành trước/sau trim và ghi nhận nhóm vector trùng. Mẫu chỉ được giữ nguyên hoặc làm sạch nhãn theo quy tắc đã nêu; không có bước chuẩn hóa, chọn đặc trưng, chia train/test hay huấn luyện. Nếu phát hiện nhãn mâu thuẫn ở một vector lặp, cell tạo `df_clean`, `X`, `y` sẽ dừng để chờ quyết định, không tự xóa mẫu.